In [1]:
import os
import json
import chromadb
from dataclasses import dataclass
from dotenv import load_dotenv
from openai import OpenAI


load_dotenv()


@dataclass
class Config:
    llm_model: str = "gpt-4o-mini"
    embed_model: str = "text-embedding-3-large"
    temperature: float = 0.0
    chroma_path: str = "./chroma_db"


config = Config()
client = OpenAI()
chroma_client = chromadb.PersistentClient(path=config.chroma_path)


def llm_json(prompt, system=None):
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    resp = client.chat.completions.create(
        model=config.llm_model,
        messages=messages,
        temperature=config.temperature,
        response_format={"type": "json_object"},
    )
    raw = resp.choices[0].message.content
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        start, end = raw.find("{"), raw.rfind("}")
        return json.loads(raw[start: end + 1])


def llm_text(prompt, system=None):
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    resp = client.chat.completions.create(
        model=config.llm_model,
        messages=messages,
        temperature=config.temperature,
    )
    return resp.choices[0].message.content.strip()


def llm_embed(text):
    try:
        resp = client.embeddings.create(
            model=config.embed_model,
            input=text,
        )
        return resp.data[0].embedding
    except Exception as e:
        print(f"[llm_embed warning] {e}")
        return None


api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    print("OPENAI_API_KEY no encontrada. Crea el .env junto al notebook.")
else:
    print(f"OpenAI key cargada (...{api_key[-6:]})")
    print(f"  LLM   : {config.llm_model}")
    print(f"  EMBED : {config.embed_model}")
    try:
        test_emb = llm_embed("hola mundo")
        if test_emb is None:
            print("  Embeddings NO disponibles (revisa la key/billing)")
        else:
            print(f"  Embeddings OK (dim={len(test_emb)})")
        test_text = llm_text("Reply with the single word: pong")
        print(f"  LLM OK -> {test_text!r}")
    except Exception as e:
        print(f"  ERROR llamando a OpenAI: {e}")

try:
    n_cols = len(chroma_client.list_collections())
    print(f"ChromaDB conectado en {config.chroma_path}  ({n_cols} colecciones existentes)")
except Exception as e:
    print("ChromaDB error:", e)


OpenAI key cargada (...OiJfEA)
  LLM   : gpt-4o-mini
  EMBED : text-embedding-3-large
  Embeddings OK (dim=3072)
  LLM OK -> 'Pong'
ChromaDB conectado en ./chroma_db  (0 colecciones existentes)


In [ ]:
n_deleted = 0
for c in chroma_client.list_collections():
    chroma_client.delete_collection(c.name)
    n_deleted += 1
print(f"ChromaDB limpio: {n_deleted} colecciones eliminadas")
print(f"Las próximas colecciones se crearán automáticamente con dim=3072")


ChromaDB limpio: 0 colecciones eliminadas
Las próximas colecciones se crearán automáticamente con dim=3072


# Fase 1 — Extracción de información

Por cada turno, el LLM extrae:
- **Entidades nombradas** (personas, animales, lugares, organizaciones, fechas, objetos)
- **Resumen factual** (preserva nombres, fechas, números)
- **Tópicos** (2-5 keywords)

Con resolución de pronombres usando el turno anterior como contexto.


In [ ]:
TITLE_PREFIXES = {
    "dr", "doctor", "mr", "mister", "mrs", "ms", "miss",
    "prof", "professor", "chef", "sir", "lady", "captain", "officer",
}


def strip_title_prefix(name):
    parts = name.split("_")
    if len(parts) > 1 and parts[0] in TITLE_PREFIXES:
        return "_".join(parts[1:])
    return name


ENTITY_EXTRACTOR_PROMPT = """You are an entity extractor for a long-term memory system.

Extract ONLY named entities that have an EXPLICIT, SPECIFIC NAME in the text.

IMPORTANT: DO NOT extract "user", "assistant", "speaker", or any conversational
role as an entity.

PRONOUN RESOLUTION:
If the current turn uses a pronoun ("his", "her", "their", "its") that clearly
refers to a named entity mentioned in the PREVIOUS turn, treat the pronoun as
that entity and extract it explicitly.

Example:
  Previous turn: "Does chef Marco run his own restaurant?"
  Current turn:  "Yes, his restaurant is called La Tavola."
  -> Extract: marco (person), la_tavola (organization)

For each entity, output:
- "name": canonical lowercase identifier with underscores.
- "type": one of [person, animal, location, organization, object, concept, event, date, attribute].

TYPE GUIDELINES:
- "person"       -> humans only with a specific name.
- "animal"       -> pets, named animals (dogs, cats, birds).
                    NEVER classify a named pet as "person".
- "location"     -> cities, neighborhoods, places, addresses.
- "organization" -> companies, restaurants, schools, shelters, clinics.
- "object"       -> specific products, brands.
- "concept"      -> named ideas (rare).
- "event"        -> specific named events.
- "date"         -> specific dates or days (Tuesdays, last Saturday, Thursday).
- "attribute"    -> rare; properties of someone (only if named).

CANONICAL NAMING RULES:
- ALWAYS strip titles ("Dr.", "Mr.", "Mrs.", "Prof.", "chef", ...).
  * "Dr. Maria Garcia" -> "maria_garcia"
  * "chef Marco"        -> "marco"
- Examples by type:
  * "the Eiffel Tower"   -> {{"name": "eiffel_tower", "type": "location"}}
  * "my brother John"    -> {{"name": "john", "type": "person"}}
  * "my dog Toby"        -> {{"name": "toby", "type": "animal"}}
  * "on Tuesdays"        -> {{"name": "tuesdays", "type": "date"}}

WHAT IS *NOT* AN ENTITY:
- Conversational roles ("user", "assistant", "I", "me", "you").
- Unnamed references.
- Generic concepts.

If no specific named entity appears, return an empty list:
{{"entities": []}}

OUTPUT — return ONLY this JSON:
{{"entities": [{{"name": "<name>", "type": "<type>"}}, ...]}}

{previous_context_section}CURRENT TURN:
---
{conversation}
---
"""


def extract_entities(text, prev_context=None):

    prev_section = ""
    if prev_context:
        prev_section = f"PREVIOUS TURN (for pronoun resolution):\n{prev_context}\n\n"

    prompt = ENTITY_EXTRACTOR_PROMPT.format(
        conversation=text,
        previous_context_section=prev_section,
    )
    result = llm_json(prompt)
    cleaned = []
    for e in result.get("entities", []):
        name = e.get("name", "").strip().lower()
        if not name:
            continue
        # Filtrar roles conversacionales y placeholders
        if name in ("user", "assistant", "speaker"):
            continue
        if name.startswith(("user_", "assistant_", "speaker_")):
            continue
        name = strip_title_prefix(name)
        if not name:
            continue
        cleaned.append({"name": name, "type": e.get("type", "concept")})
    return cleaned


In [4]:
SUMMARY_PROMPT = """You are a memory system processing a conversation turn.

Generate a 1-2 sentence summary (third-person, max ~30 words) that PRESERVES
all specific factual values: numbers, money, times, dates, names, brands.

CRITICAL RULES:
1. SPEAKER ATTRIBUTION — must start with "The {speaker}":
   - speaker = "user"      -> "The user lives in..."
   - speaker = "assistant" -> "The assistant said..."

2. ONLY USE INFORMATION FROM THE CURRENT TURN.

3. PRONOUN RESOLUTION: if the current turn uses pronouns referring to a named
   entity in the PREVIOUS TURN, EXPAND the pronoun to the explicit name.

   Example:
     Previous turn: "Does chef Marco run his own restaurant?"
     Current turn:  "Yes, his restaurant is called La Tavola."
     GOOD: "The user confirmed Marco's restaurant is called La Tavola."

4. SHORT TURNS GET SHORT SUMMARIES.

Also extract 2-5 short lowercase topics relevant to the current turn.

Speaker: {speaker}

{previous_context_section}OUTPUT FORMAT — return ONLY this JSON:
{{"summary": "<factual summary starting with 'The {speaker}'>", "topics": ["<t1>", "<t2>"]}}

CURRENT TURN:
---
{turn}
---
"""


def extract_summary_topics(text, speaker, prev_context=None):
    prev_section = ""
    if prev_context:
        prev_section = f"PREVIOUS TURN (for pronoun resolution):\n{prev_context}\n\n"

    prompt = SUMMARY_PROMPT.format(
        turn=text,
        speaker=speaker,
        previous_context_section=prev_section,
    )
    result = llm_json(prompt)
    return {
        "summary": result.get("summary", text[:80]),
        "topics": result.get("topics", []),
    }


In [ ]:
def phase1_extract(text, speaker, prev_context=None):

    entities = extract_entities(text, prev_context=prev_context)
    st = extract_summary_topics(text, speaker=speaker, prev_context=prev_context)
    return {
        "entities": entities,
        "summary": st["summary"],
        "topics": st["topics"],
    }


# Fase 2 — Actualización del grafo

1. **Inserción**: cada turno crea un nodo y se conecta vía `MENTIONS` a sus entidades.
2. **Conflict resolution (Mem0g)**: el LLM decide *ADD / MODIFY / KEEP* sobre los
   atributos de cada entidad. Se ejecuta tanto al CREAR la entidad como al re-encontrarla.
3. **Recálculo de relevancia**:



In [6]:
# ADD    : el turno aporta info NUEVA -> añadir a attributes
# MODIFY : el turno CONTRADICE o ACTUALIZA info previa -> reemplazar
# KEEP   : el turno no aporta info estructural -> no tocar attributes

ENTITY_CONFLICT_PROMPT = """You are a memory system maintaining structured attributes for named entities.

Given:
  - An entity already in memory with its CURRENT attributes (may be empty)
  - A NEW turn that mentions this entity

Decide ONE action:
  - "ADD"    : new turn reveals NEW facts about this entity -> add to attributes.
  - "MODIFY" : new turn CONTRADICTS or UPDATES existing facts -> replace values.
  - "KEEP"   : no new factual attribute, just re-mention -> keep unchanged.

Attribute values must be CONCISE strings (1-5 words).

Examples:

  Entity: "toby"  | Attributes: {{}}
  Turn: "I adopted Toby last Saturday."
  -> {{"action": "ADD", "attributes": {{"adopted_on": "last Saturday"}}}}

  Entity: "toby"  | Attributes: {{"adopted_on": "last Saturday"}}
  Turn: "Toby is a two-year-old beagle from Refugio Esperanza."
  -> {{"action": "ADD", "attributes": {{"adopted_on": "last Saturday", "age": "2 years", "breed": "beagle", "origin": "Refugio Esperanza"}}}}

  Entity: "toby"  | Attributes: {{"age": "2 years", "breed": "beagle"}}
  Turn: "Actually Toby just turned 3 years old."
  -> {{"action": "MODIFY", "attributes": {{"age": "3 years", "breed": "beagle"}}}}

  Entity: "toby"  | Attributes: {{"breed": "beagle"}}
  Turn: "I took Toby to the beach today."
  -> {{"action": "KEEP", "attributes": {{"breed": "beagle"}}}}

  Entity: "glovo" | Attributes: {{}}
  Turn: "It is at a startup called Glovo and I work as a backend engineer."
  -> {{"action": "ADD", "attributes": {{"type": "startup", "user_role": "backend engineer"}}}}

ENTITY: {entity_name}
CURRENT ATTRIBUTES: {current_attributes}

NEW TURN (speaker={speaker}):
---
{turn_text}
---

Output ONLY this JSON:
{{"action": "ADD" | "MODIFY" | "KEEP", "attributes": {{...COMPLETE new attributes dict...}}}}
"""


def resolve_entity_conflict(entity_name, current_attributes, turn_text, speaker):
    try:
        prompt = ENTITY_CONFLICT_PROMPT.format(
            entity_name=entity_name,
            current_attributes=json.dumps(current_attributes),
            speaker=speaker,
            turn_text=turn_text,
        )
        result = llm_json(prompt)
        action = result.get("action", "KEEP")
        attrs = result.get("attributes", current_attributes)
        if action not in ("ADD", "MODIFY", "KEEP"):
            action = "KEEP"
            attrs = current_attributes
        if not isinstance(attrs, dict):
            attrs = current_attributes
        return {"action": action, "attributes": attrs}
    except Exception as e:
        print(f"[resolve_entity_conflict warning] {entity_name}: {e}")
        return {"action": "KEEP", "attributes": current_attributes}


In [ ]:
import math
import networkx as nx
from datetime import datetime, timezone


def now_iso():
    return datetime.now(timezone.utc).isoformat()


def _build_embed_text(summary, topics):

    topics_str = ", ".join(topics) if topics else ""
    return f"{summary}\nTopics: {topics_str}" if topics_str else summary


class ConvMemoryGraph:


    def __init__(self,
                 alpha=0.3, beta=0.4, gamma=0.3, lam=0.05, n_max=30,
                 compute_embeddings=True,
                 enable_conflict_resolution=True,
                 collection_name="default",
                 reset_collection=False):
        self.g = nx.MultiDiGraph()
        self.turn_counter = 0
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
        self.lam = lam
        self.n_max = n_max
        self.compute_embeddings = compute_embeddings
        self.enable_conflict_resolution = enable_conflict_resolution
        self.collection_name = collection_name
        self._prev_statement = None

        if compute_embeddings:
            if reset_collection:
                try:
                    chroma_client.delete_collection(collection_name)
                except Exception:
                    pass
            self.collection = chroma_client.get_or_create_collection(
                name=collection_name,
                metadata={"hnsw:space": "cosine"},
            )
        else:
            self.collection = None

    # ------------------------------------------------------------------
    # Inserción de turnos
    # ------------------------------------------------------------------
    def add(self, text, speaker="user", kind="statement"):
        # Contexto del turno anterior para resolución de pronombres
        prev_context = None
        if self._prev_statement is not None:
            prev_sp, prev_txt = self._prev_statement
            prev_context = f"{prev_sp}: {prev_txt}"

        # Fase 1
        ext = phase1_extract(text, speaker, prev_context=prev_context)
        summary = ext["summary"]
        topics = ext["topics"]

        if self.compute_embeddings:
            embedding = llm_embed(_build_embed_text(summary, topics))
        else:
            embedding = None

        position = self.turn_counter
        turn_id = f"t{position}"
        self.g.add_node(
            turn_id,
            node_type="turn",
            position=position,
            summary=summary,
            topics=topics,
            role=speaker,
            kind=kind,
            is_query=(kind == "query"),
            r=1.0,
            created_at=now_iso(),
        )

        if (embedding is not None
                and kind == "statement"
                and self.collection is not None):
            try:
                self.collection.add(
                    ids=[turn_id],
                    embeddings=[embedding],
                    metadatas=[{
                        "position": position,
                        "role": speaker,
                        "kind": kind,
                    }],
                )
            except Exception as e:
                print(f"[chroma add warning] {turn_id}: {e}")

        # Fase 2: nodos entidad + aristas MENTIONS + conflict resolution
        n_conflict_calls = 0
        for e in ext["entities"]:
            name, etype = e["name"], e["type"]
            is_new = name not in self.g.nodes

            if is_new:
                self.g.add_node(
                    name,
                    node_type="entity",
                    entity_type=etype,
                    attributes={},
                    first_seen=position,
                    last_seen=position,
                    created_at=now_iso(),
                )
                # Conflict resolver al CREAR: nueva entidad nace con atributos
                if self.enable_conflict_resolution and kind != "query":
                    resolution = resolve_entity_conflict(
                        entity_name=name,
                        current_attributes={},
                        turn_text=text,
                        speaker=speaker,
                    )
                    n_conflict_calls += 1
                    if resolution["action"] != "KEEP" and resolution["attributes"]:
                        self.g.nodes[name]["attributes"] = resolution["attributes"]
            else:
                # Conflict resolver al RE-ENCONTRAR: actualiza atributos
                if self.enable_conflict_resolution and kind != "query":
                    current_attrs = self.g.nodes[name].get("attributes", {})
                    resolution = resolve_entity_conflict(
                        entity_name=name,
                        current_attributes=current_attrs,
                        turn_text=text,
                        speaker=speaker,
                    )
                    n_conflict_calls += 1
                    if resolution["action"] != "KEEP":
                        self.g.nodes[name]["attributes"] = resolution["attributes"]
                self.g.nodes[name]["last_seen"] = position

            self.g.add_edge(turn_id, name,
                            edge_type="MENTIONS",
                            created_at=now_iso())

        # Recalcular r(t_i) para todos los turnos
        t_actual = position
        for tid in self._turn_ids():
            self.g.nodes[tid]["r"] = self._compute_r(tid, t_actual)

        # Poda condicional si excedemos n_max
        n_pruned = 0
        n_orphans = 0
        if self._n_turns() > self.n_max:
            n_pruned = self._prune_low_relevance_turns()
            n_orphans = self._prune_orphan_entities()

        # Actualizar buffer para próxima llamada (solo statements)
        if kind == "statement":
            self._prev_statement = (speaker, text)

        self.turn_counter += 1
        return {
            "turn_id": turn_id,
            "position": position,
            "role": speaker,
            "kind": kind,
            "n_entities": len(ext["entities"]),
            "n_conflict_calls": n_conflict_calls,
            "embedding": embedding,
            "n_pruned": n_pruned,
            "n_orphans_removed": n_orphans,
        }

    # r(t_i) = α·ant + β·men + γ·ult

    def _compute_r(self, turn_id, t_actual):
        i = self.g.nodes[turn_id]["position"]

        # Antigüedad: decae lentamente con lambda=0.05
        ant = math.exp(-self.lam * (t_actual - i))

        # Menciones: cuántos turnos comparten alguna entidad con t_i
        entities_i = self._entities_of_turn(turn_id)
        sharing_count = 0
        last_mention_j = i

        for tid in self._turn_ids():
            j = self.g.nodes[tid]["position"]
            if j < i:
                continue
            if j == i:
                sharing_count += 1
                continue
            entities_j = self._entities_of_turn(tid)
            if entities_i & entities_j:
                sharing_count += 1
                last_mention_j = max(last_mention_j, j)

        men = sharing_count / max(1, t_actual + 1)

        # Última aparición: decae con la distancia a la última mención
        ult = math.exp(-self.lam * (t_actual - last_mention_j))

        return self.alpha * ant + self.beta * men + self.gamma * ult

    # ------------------------------------------------------------------
    # Helpers
    # ------------------------------------------------------------------
    def _turn_ids(self):
        return [n for n, d in self.g.nodes(data=True)
                if d.get("node_type") == "turn"]

    def _entity_ids(self):
        return [n for n, d in self.g.nodes(data=True)
                if d.get("node_type") == "entity"]

    def _n_turns(self):
        return len(self._turn_ids())

    def _entities_of_turn(self, turn_id):
        """Entidades mencionadas por un turno."""
        out = set()
        for _, v, d in self.g.out_edges(turn_id, data=True):
            if d.get("edge_type") == "MENTIONS":
                out.add(v)
        return out

    def _turns_mentioning_entity(self, entity_name, include_queries=False):
        """Turnos que mencionan una entidad."""
        if entity_name not in self.g.nodes:
            return []
        turns = []
        for u, _, d in self.g.in_edges(entity_name, data=True):
            if d.get("edge_type") != "MENTIONS":
                continue
            node = self.g.nodes[u]
            if node.get("node_type") != "turn":
                continue
            if not include_queries and node.get("is_query"):
                continue
            turns.append(u)
        return turns

    def _prune_low_relevance_turns(self):
        """Elimina turnos con menor r(t_i) hasta volver a n_max. Borra de Chroma."""
        n_to_remove = self._n_turns() - self.n_max
        if n_to_remove <= 0:
            return 0
        turns_sorted = sorted(self._turn_ids(),
                              key=lambda tid: self.g.nodes[tid]["r"])
        to_delete_ids = [tid for tid in turns_sorted[:n_to_remove]]
        for tid in to_delete_ids:
            self.g.remove_node(tid)
        if self.collection is not None and to_delete_ids:
            try:
                self.collection.delete(ids=to_delete_ids)
            except Exception as e:
                print(f"[chroma prune warning] {e}")
        return n_to_remove

    def _prune_orphan_entities(self):
        """Elimina entidades sin aristas MENTIONS y sin atributos."""
        to_remove = [
            eid for eid in self._entity_ids()
            if self.g.in_degree(eid) == 0
            and not self.g.nodes[eid].get("attributes")
        ]
        for eid in to_remove:
            self.g.remove_node(eid)
        return len(to_remove)

    def search_similar_statements(self, query_embedding, n_results=20,
                                  exclude_position_gte=None):
        """Devuelve [(turn_id, similarity), ...] usando ChromaDB."""
        if self.collection is None or query_embedding is None:
            return []
        count = self.collection.count()
        if count == 0:
            return []
        if exclude_position_gte is not None:
            where = {
                "$and": [
                    {"kind": {"$eq": "statement"}},
                    {"position": {"$lt": exclude_position_gte}},
                ]
            }
        else:
            where = {"kind": {"$eq": "statement"}}
        try:
            results = self.collection.query(
                query_embeddings=[query_embedding],
                n_results=min(n_results, count),
                where=where,
                include=["distances"],
            )
        except Exception as e:
            print(f"[chroma query warning] {e}")
            return []
        ids = results.get("ids", [[]])[0]
        dists = results.get("distances", [[]])[0]
        return [(tid, max(0.0, 1.0 - d)) for tid, d in zip(ids, dists)]


    def show_state(self):
        n_t = self._n_turns()
        n_e = len(self._entity_ids())
        n_chroma = self.collection.count() if self.collection else 0
        print(f"\n{'='*72}")
        print(f"GRAFO  |  turnos: {n_t}  |  entidades: {n_e}  |  chroma: {n_chroma}")
        print('='*72)
        for tid in sorted(self._turn_ids(),
                          key=lambda x: self.g.nodes[x]["position"]):
            d = self.g.nodes[tid]
            menciona = sorted(self._entities_of_turn(tid))
            kind_tag = " [QUERY]" if d.get("is_query") else ""
            print(f"  [{tid}] pos={d['position']}  role={d.get('role','?')}{kind_tag}  "
                  f"r={d['r']:.3f}")
            print(f"        {d['summary']}")
            print(f"        menciona: {menciona}\n")

    def show_state_compact(self, last_n=None):
        n_t = self._n_turns()
        n_e = len(self._entity_ids())
        n_chroma = self.collection.count() if self.collection else 0
        print(f"\n{'─'*72}")
        print(f"GRAFO  |  turnos: {n_t}  |  entidades: {n_e}  |  chroma: {n_chroma}")
        print('─'*72)
        tids = sorted(self._turn_ids(),
                      key=lambda x: self.g.nodes[x]["position"])
        if last_n is not None and len(tids) > last_n:
            print(f"  ... ({len(tids) - last_n} turnos anteriores omitidos)")
            tids = tids[-last_n:]
        for tid in tids:
            d = self.g.nodes[tid]
            tag = "Q" if d.get("is_query") else "S"
            print(f"  [{tid:>4}] pos={d['position']:>2} {tag} "
                  f"role={d.get('role','?'):<9} r={d['r']:.2f}  "
                  f"{d['summary'][:55]}")


# Fase 3 — Recuperación de contexto + Respuesta del LLM




In [ ]:
ANSWER_PROMPT_C1 = """You are a memory assistant answering questions about a user.

The MEMORIES below are summaries of past turns, sorted in REVERSE CHRONOLOGICAL ORDER.

Each turn may include a "Mentions:" section listing named entities with their
structured attributes. Use those facts when answering questions about that entity.

ANSWER RULES:
1. Be CONCISE — output only the answer, no preamble.
2. Match the answer to the question's intent:
   - "Where does X live?" -> a geographic place.
   - "Where does X work?" -> an EMPLOYER (company), NOT a city.
   - "Who ..." -> a person's name.
   - "How many / how much ..." -> a number or amount.
   - "When ..." -> a date or time.
3. CONNECT MEMORIES: combine facts across turns when needed.
4. RECENCY: if memories conflict, trust the more recent (higher position).
5. NEVER output snake_case identifiers (placeholders). Rephrase naturally.
6. Only say "I don't know" if memories truly lack the info.

MEMORIES (MOST RECENT FIRST):
{context}

QUESTION: {query}

ANSWER:"""


def build_context(mem, turn_ids):
    """Contexto al LLM con atributos estructurados de Mem0g."""
    sorted_tids = sorted(turn_ids,
                         key=lambda t: mem.g.nodes[t]["position"],
                         reverse=True)
    lines = []
    for tid in sorted_tids:
        d = mem.g.nodes[tid]
        lines.append(f"[Turn {d['position']} | role={d.get('role','?')} | r={d['r']:.2f}]")
        lines.append(f"Summary: {d['summary']}")
        mentions = sorted(mem._entities_of_turn(tid))
        if mentions:
            lines.append("Mentions:")
            for m in mentions:
                ent_node = mem.g.nodes[m]
                etype = ent_node.get("entity_type", "?")
                attrs = ent_node.get("attributes", {}) or {}
                if attrs:
                    attr_str = ", ".join(f"{k}={v}" for k, v in attrs.items())
                    lines.append(f"  - {m} ({etype}): {attr_str}")
                else:
                    lines.append(f"  - {m} ({etype})")
        lines.append("")
    return "\n".join(lines).strip()


# Recuperación read-only sobre ChromaDB
def retrieve_relevant_turns(mem, query, speaker="user",
                            k=10, w_sim=0.7, w_r=0.3,
                            chroma_topn=50):

    # Read-only: calcular embedding sin tocar el grafo
    ext = phase1_extract(query, speaker)
    q_emb = llm_embed(_build_embed_text(ext["summary"], ext["topics"]))
    q_pos = mem.turn_counter  # posición virtual (sin añadir nada)

    chroma_hits = mem.search_similar_statements(
        query_embedding=q_emb,
        n_results=chroma_topn,
        exclude_position_gte=q_pos,
    )

    breakdown = {}
    for tid, sim in chroma_hits:
        if tid not in mem.g.nodes:
            continue
        r_val = mem.g.nodes[tid].get("r", 0.0)
        score = w_sim * sim + w_r * r_val
        breakdown[tid] = (score, sim, r_val)

    ranked = sorted(breakdown.keys(),
                    key=lambda t: breakdown[t][0], reverse=True)

    q_info = {
        "summary": ext["summary"],
        "topics": ext["topics"],
        "entities": [e["name"] for e in ext["entities"]],
    }
    return q_info, ranked[:k], breakdown


def answer(mem, query, speaker="user",
           k=10, w_sim=0.7, w_r=0.3,
           chroma_topn=50,
           verbose=False, return_meta=False):
    q_info, top_turns, breakdown = retrieve_relevant_turns(
        mem, query, speaker=speaker,
        k=k, w_sim=w_sim, w_r=w_r,
        chroma_topn=chroma_topn,
    )
    if not top_turns:
        text = "I don't know — no memories available yet."
    else:
        context = build_context(mem, top_turns)
        prompt = ANSWER_PROMPT_C1.format(context=context, query=query)
        text = llm_text(prompt)

    if verbose:
        print("=== INFO DE LA PREGUNTA (no añadida al grafo) ===")
        print(f"  resumen: {q_info['summary']}")
        print(f"  entidades: {q_info['entities']}")
        print(f"  pesos: w_sim={w_sim}, w_r={w_r}, k={k}")
        print(f"  Chroma: {len(breakdown)} candidatos (topn={chroma_topn})")

        print(f"\n=== TOP-{len(top_turns)} (score híbrido sim+r) ===")
        for tid in top_turns:
            score, sim, r_val = breakdown[tid]
            summary = mem.g.nodes[tid]["summary"]
            print(f"  {tid}  score={score:.3f}  (sim={sim:.3f}, r={r_val:.3f})  {summary[:55]}")

        print("\n=== CONTEXTO ENVIADO AL LLM ===")
        print(build_context(mem, top_turns) if top_turns else "(vacío)")
        print(f"\nQUESTION: {query}")
        print("=" * 50)

    if return_meta:
        return {"text": text, "top_turn_ids": list(top_turns), "breakdown": breakdown}
    return text


# LongMemEval — Loader y evaluación

Pipeline:
1. **Loader**: lee `longmemeval_oracle.json` (500 preguntas).
2. **`process_question(qdata)`**: por cada pregunta crea un grafo nuevo
   (colección Chroma dedicada), ingiere todas las sesiones del *haystack*
   como statements, ejecuta `answer(...)` y limpia la colección.
3. **Smoke test**: una sola pregunta con `verbose=True` para verificar
   recuperación y respuesta antes de lanzar el loop completo.

Variante usada: **oracle** (sólo trae las sesiones que contienen la
respuesta — ideal para validar el pipeline antes de pasar a *small / medium*).


In [ ]:
import os
from collections import Counter

LONGMEM_PATH = r"C:\Users\HIlla\Documents\imple2\LongMemEval\data\longmemeval_oracle.json"


def load_longmemeval(path=LONGMEM_PATH):
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
    return data


def lme_stats(dataset):
    n = len(dataset)
    types = Counter(q["question_type"] for q in dataset)
    n_sessions = [len(q["haystack_sessions"]) for q in dataset]
    n_turns = [sum(len(s) for s in q["haystack_sessions"]) for q in dataset]
    print(f"Dataset: {os.path.basename(LONGMEM_PATH)}")
    print(f"  preguntas         : {n}")
    print(f"  sesiones / preg.  : min={min(n_sessions)}  max={max(n_sessions)}  "
          f"avg={sum(n_sessions)/n:.1f}")
    print(f"  turnos   / preg.  : min={min(n_turns)}  max={max(n_turns)}  "
          f"avg={sum(n_turns)/n:.1f}")
    print(f"  tipos:")
    for t, c in types.most_common():
        print(f"     {t:35s} {c}")


lme_data = load_longmemeval()
lme_stats(lme_data)

q0 = lme_data[0]
print("\n\n--- EJEMPLO Q0 ---")
print(f"  question_id   : {q0['question_id']}")
print(f"  question_type : {q0['question_type']}")
print(f"  question      : {q0['question']}")
print(f"  gold answer   : {q0['answer']}")
print(f"  n sessions    : {len(q0['haystack_sessions'])}")
print(f"  n turns total : {sum(len(s) for s in q0['haystack_sessions'])}")
print(f"  answer_session_ids : {q0.get('answer_session_ids')}")


Dataset: longmemeval_oracle.json
  preguntas         : 500
  sesiones / preg.  : min=1  max=6  avg=1.9
  turnos   / preg.  : min=2  max=72  avg=21.9
  tipos:
     temporal-reasoning                  133
     multi-session                       133
     knowledge-update                    78
     single-session-user                 70
     single-session-assistant            56
     single-session-preference           30


--- EJEMPLO Q0 ---
  question_id   : gpt4_2655b836
  question_type : temporal-reasoning
  question      : What was the first issue I had with my new car after its first service?
  gold answer   : GPS system not functioning correctly
  n sessions    : 3
  n turns total : 36
  answer_session_ids : ['answer_4be1b6b4_2', 'answer_4be1b6b4_3', 'answer_4be1b6b4_1']


In [10]:
import time
import re


def _safe_collection_name(qid):
    name = re.sub(r"[^a-zA-Z0-9_.-]", "_", f"lme_{qid}")
    return name[:60].strip("._-") or "lme_q"


def process_question(qdata,
                    verbose=False,
                    k=10,
                    n_max=400,
                    enable_conflict_resolution=True,
                    cleanup_collection=True):
    """Procesa 1 pregunta de LongMemEval y devuelve dict con las 3 métricas:
        - 'correct'           : accuracy (juez aplica fuera)
        - 'recall_at_k'       : 1 si algún top-k pertenece a answer_session_ids
        - 't_ingest_s'        : latencia ingesta total
        - 't_answer_s'        : latencia query (retrieve + answer)
        - 'latency_per_turn_s': t_ingest / n_turns
        - 'latency_query_s'   : alias de t_answer_s
        - 'top_turn_ids'      : top-k recuperados (para análisis posterior)
    """
    qid = qdata["question_id"]
    coll_name = _safe_collection_name(qid)
    n_sessions = len(qdata["haystack_sessions"])
    n_turns = sum(len(s) for s in qdata["haystack_sessions"])
    answer_session_ids = set(qdata.get("answer_session_ids", []))
    haystack_session_ids = qdata.get("haystack_session_ids", [])

    if verbose:
        print(f"[{qid}] type={qdata['question_type']}  "
              f"sessions={n_sessions}  turns={n_turns}")
        print(f"        collection='{coll_name}'  n_max={n_max}  "
              f"conflict_res={enable_conflict_resolution}  k={k}")
        print(f"        answer_session_ids={list(answer_session_ids)}")

    mem = ConvMemoryGraph(
        alpha=0.3, beta=0.4, gamma=0.3, lam=0.05,
        n_max=n_max,
        collection_name=coll_name,
        reset_collection=True,
        enable_conflict_resolution=enable_conflict_resolution,
    )

    # Ingesta
    turn_to_session = {}
    t0 = time.time()
    for si, session in enumerate(qdata["haystack_sessions"]):
        sid = haystack_session_ids[si] if si < len(haystack_session_ids) else f"sess_{si}"
        for turn in session:
            role = turn.get("role", "user")
            content = turn.get("content", "")
            if not content.strip():
                continue
            r = mem.add(content, speaker=role, kind="statement")
            turn_to_session[r["turn_id"]] = sid
        if verbose:
            print(f"        session {si+1}/{n_sessions} ingestada "
                  f"(turnos grafo: {mem._n_turns()})")
    t_ingest = time.time() - t0

    # Pregunta
    t1 = time.time()
    ans_meta = answer(mem, qdata["question"], speaker="user",
                      k=k, verbose=verbose, return_meta=True)
    t_answer = time.time() - t1
    pred = ans_meta["text"]
    top_ids = ans_meta["top_turn_ids"]

    # Recall@k a nivel sesión
    retrieved_sessions = {turn_to_session.get(tid) for tid in top_ids}
    retrieved_sessions.discard(None)
    if answer_session_ids:
        recall_at_k = 1 if (retrieved_sessions & answer_session_ids) else 0
    else:
        recall_at_k = None

    # Latencias
    n_turns_real = max(1, len(turn_to_session))
    latency_per_turn = t_ingest / n_turns_real
    latency_query = t_answer

    if verbose:
        print(f"\n        GOLD : {qdata['answer']}")
        print(f"        PRED : {pred}")
        print(f"        top_turn_ids: {top_ids}")
        print(f"        retrieved_sessions: {sorted(retrieved_sessions)}")
        print(f"        recall_at_k: {recall_at_k}")
        print(f"        latency_per_turn: {latency_per_turn:.2f}s  "
              f"latency_query: {latency_query:.2f}s")

    if cleanup_collection:
        try:
            chroma_client.delete_collection(coll_name)
        except Exception:
            pass

    return {
        "question_id": qid,
        "question_type": qdata["question_type"],
        "question": qdata["question"],
        "gold": qdata["answer"],
        "prediction": pred,
        "n_sessions": n_sessions,
        "n_turns": n_turns,
        "t_ingest_s": round(t_ingest, 2),
        "t_answer_s": round(t_answer, 2),
        "latency_per_turn_s": round(latency_per_turn, 3),
        "latency_query_s": round(latency_query, 3),
        "top_turn_ids": top_ids,
        "retrieved_sessions": sorted(s for s in retrieved_sessions if s),
        "recall_at_k": recall_at_k,
    }


# Evaluación con métricas (LLM-as-judge)

- **Accuracy global** y por `question_type` (6 tipos).
- **Recall@k**: 1 si algún turno del top-k pertenece a una `answer_session_id`.
- **Latencia**: tiempo de ingesta por turno + tiempo de query.


In [11]:
# ----------------------------------------------------------------------
# LLM-as-judge: gpt-4o-mini decide si PRED es correcta vs GOLD
# ----------------------------------------------------------------------
JUDGE_PROMPT = """You are an impartial judge for a memory QA system.

Given a QUESTION, the GOLD ANSWER, and a PREDICTED ANSWER, decide if the
predicted answer is CORRECT.

Be LENIENT with formatting: the prediction may be a full sentence while the
gold is a short fact. What matters is whether the prediction conveys the
SAME FACTUAL ANSWER as the gold.

Be STRICT with content: a vague or "I don't know" answer is wrong, even if
the topic is right.

Examples:

  GOLD: "GPS system not functioning correctly"
  PRED: "The first issue was a problem with the GPS system."
  -> YES (same fact, just rephrased)

  GOLD: "8 kilometers"
  PRED: "About 8 km"
  -> YES (same number)

  GOLD: "Glovo"
  PRED: "He works at Glovo, a delivery startup."
  -> YES (mentions Glovo)

  GOLD: "Saturday"
  PRED: "Last weekend"
  -> NO (less specific)

  GOLD: "Refugio Esperanza"
  PRED: "An animal shelter"
  -> NO (correct concept but not the name)

  GOLD: "GPS system not functioning correctly"
  PRED: "I don't know."
  -> NO (no answer given)

  GOLD: "Yes"
  PRED: "Based on the memories, yes."
  -> YES (same verdict)

QUESTION: {question}
GOLD: {gold}
PRED: {pred}

Reply with ONLY this JSON:
{{"verdict": "YES" | "NO", "reason": "<one short sentence>"}}
"""


def judge_answer(question, gold, pred):
    """Devuelve {correct: bool, verdict: 'YES'/'NO', reason: str}."""
    if not pred or not str(pred).strip():
        return {"correct": False, "verdict": "NO", "reason": "Empty prediction"}
    try:
        result = llm_json(JUDGE_PROMPT.format(
            question=question,
            gold=gold,
            pred=pred,
        ))
        verdict = str(result.get("verdict", "NO")).upper().strip()
        return {
            "correct": verdict == "YES",
            "verdict": verdict,
            "reason": result.get("reason", "")[:200],
        }
    except Exception as e:
        return {"correct": False, "verdict": "ERROR", "reason": str(e)[:200]}


In [ ]:
# ----------------------------------------------------------------------
# SMOKE TEST: 1 pregunta (Q0) end-to-end con verbose.

qdata_q0 = lme_data[0]
result_q0 = process_question(
    qdata_q0,
    verbose=True,
    k=10,
    n_max=400,
    enable_conflict_resolution=True,
)

# Juez automático
jud_q0 = judge_answer(qdata_q0["question"], qdata_q0["answer"], result_q0["prediction"])
result_q0["correct"] = jud_q0["correct"]
result_q0["judge_verdict"] = jud_q0["verdict"]
result_q0["judge_reason"] = jud_q0["reason"]

print("\n" + "="*72)
print("SMOKE TEST Q0 — RESULTADO")
print("="*72)
for k_, v in result_q0.items():
    if isinstance(v, str) and len(v) > 100:
        v = v[:100] + "..."
    print(f"  {k_:20s} : {v}")

if result_q0["correct"]:
    print("\n>>> Q0 CORRECTO. Sistema listo para lanzar eval-100.")
else:
    print("\n>>> Q0 INCORRECTO. Revisa el contexto antes del eval-100.")


[gpt4_2655b836] type=temporal-reasoning  sessions=3  turns=36
        collection='lme_gpt4_2655b836'  n_max=400  conflict_res=True  k=10
        answer_session_ids=['answer_4be1b6b4_2', 'answer_4be1b6b4_3', 'answer_4be1b6b4_1']
        session 1/3 ingestada (turnos grafo: 12)
        session 2/3 ingestada (turnos grafo: 24)
        session 3/3 ingestada (turnos grafo: 36)
=== INFO DE LA PREGUNTA (no añadida al grafo) ===
  resumen: The user asked about the first issue they had with their new car after its first service.
  entidades: []
  pesos: w_sim=0.7, w_r=0.3, k=10
  Chroma: 36 candidatos (topn=50)

=== TOP-10 (score híbrido sim+r) ===
  t16  score=0.450  (sim=0.469, r=0.405)  The user plans to ask the detailer questions about thei
  t34  score=0.447  (sim=0.390, r=0.582)  The user is considering getting a car wax and detailing
  t15  score=0.437  (sim=0.525, r=0.232)  The assistant provided information on common GPS issues
  t24  score=0.435  (sim=0.434, r=0.440)  The user is plan

In [ ]:
import random
from collections import defaultdict, Counter
from tqdm.auto import tqdm


def select_balanced_subset(dataset, n_total=30, seed=42):
    """Selecciona n_total preguntas balanceadas por question_type."""
    rng = random.Random(seed)
    by_type = defaultdict(list)
    for q in dataset:
        by_type[q["question_type"]].append(q)

    types = sorted(by_type.keys())
    per_type = max(1, n_total // len(types))

    selected = []
    for t in types:
        bucket = by_type[t][:]
        rng.shuffle(bucket)
        selected.extend(bucket[:per_type])

    remaining = n_total - len(selected)
    if remaining > 0:
        all_remaining = [q for t in types for q in by_type[t][per_type:]]
        rng.shuffle(all_remaining)
        selected.extend(all_remaining[:remaining])

    rng.shuffle(selected)
    return selected[:n_total]


def run_eval(questions,
             output_path="eval_results.json",
             k=5,
             n_max=400,
             enable_conflict_resolution=True,
             resume=True):
    """Loop con checkpoint a disco. Tras cada pregunta guarda el JSON."""
    results = {}
    if resume and os.path.exists(output_path):
        try:
            with open(output_path, encoding="utf-8") as f:
                existing = json.load(f)
            for r in existing:
                results[r["question_id"]] = r
            print(f"[resume] {len(results)} preguntas ya procesadas en '{output_path}'")
        except Exception as e:
            print(f"[resume warning] no se pudo leer {output_path}: {e}")

    pending = [q for q in questions if q["question_id"] not in results]
    print(f"[eval] total={len(questions)}  ya hechas={len(results)}  "
          f"pendientes={len(pending)}")
    if not pending:
        print("Nada que hacer.")
        return list(results.values())

    pbar = tqdm(pending, desc="Eval", unit="q")
    for qdata in pbar:
        qid = qdata["question_id"]
        try:
            res = process_question(
                qdata,
                verbose=False,
                k=k, n_max=n_max,
                enable_conflict_resolution=enable_conflict_resolution,
            )
            jud = judge_answer(qdata["question"], qdata["answer"], res["prediction"])
            res["correct"] = jud["correct"]
            res["judge_verdict"] = jud["verdict"]
            res["judge_reason"] = jud["reason"]
            results[qid] = res
        except Exception as e:
            print(f"\n[error] {qid}: {e}")
            results[qid] = {
                "question_id": qid,
                "question_type": qdata.get("question_type"),
                "question": qdata.get("question"),
                "gold": qdata.get("answer"),
                "prediction": None,
                "correct": False,
                "recall_at_k": None,
                "judge_verdict": "ERROR",
                "judge_reason": str(e)[:200],
            }

        try:
            with open(output_path, "w", encoding="utf-8") as f:
                json.dump(list(results.values()), f, ensure_ascii=False, indent=2)
        except Exception as e:
            print(f"\n[checkpoint warning] {e}")

        done = list(results.values())
        n_ok = sum(1 for r in done if r.get("correct"))
        acc = n_ok / len(done) * 100
        rec_vals = [r.get("recall_at_k") for r in done if r.get("recall_at_k") is not None]
        rec = (sum(rec_vals) / len(rec_vals) * 100) if rec_vals else 0.0
        pbar.set_postfix(acc=f"{acc:.1f}%", rec=f"{rec:.1f}%", n=len(done))

    return list(results.values())


def print_stats(results):

    if not results:
        print("(sin resultados)")
        return
    total = len(results)
    n_correct = sum(1 for r in results if r.get("correct"))
    accuracy = n_correct / total

    rec_vals = [r.get("recall_at_k") for r in results if r.get("recall_at_k") is not None]
    recall = (sum(rec_vals) / len(rec_vals)) if rec_vals else 0.0
    n_rec = len(rec_vals)

    lat_turn_vals = [r.get("latency_per_turn_s") for r in results
                     if r.get("latency_per_turn_s") is not None]
    lat_query_vals = [r.get("latency_query_s") for r in results
                      if r.get("latency_query_s") is not None]
    lat_turn = sum(lat_turn_vals) / len(lat_turn_vals) if lat_turn_vals else 0.0
    lat_query = sum(lat_query_vals) / len(lat_query_vals) if lat_query_vals else 0.0

    print(f"\n{'='*66}")
    print(f"MÉTRICAS  ({total} preguntas)")
    print(f"{'='*66}")
    print(f"  1) Accuracy (LLM-as-judge) : {accuracy*100:5.1f}%  ({n_correct}/{total})")
    print(f"  2) Recall@k del retrieve   : {recall*100:5.1f}%  ({n_rec} preg. con label)")
    print(f"  3) Latencia                : ingest/turno = {lat_turn:5.2f}s  "
          f"|  query = {lat_query:5.2f}s")

    by_type = defaultdict(lambda: {"correct": 0, "recall": [], "total": 0,
                                     "lat_turn": [], "lat_query": []})
    for r in results:
        t = r.get("question_type", "unknown")
        s = by_type[t]
        s["total"] += 1
        if r.get("correct"):
            s["correct"] += 1
        if r.get("recall_at_k") is not None:
            s["recall"].append(r["recall_at_k"])
        if r.get("latency_per_turn_s") is not None:
            s["lat_turn"].append(r["latency_per_turn_s"])
        if r.get("latency_query_s") is not None:
            s["lat_query"].append(r["latency_query_s"])

    print(f"\nDesglose por question_type:")
    print(f"  {'tipo':<32s}  {'acc':>6s}  {'rec@k':>6s}  {'lat/turn':>8s}  {'lat/q':>6s}")
    for t in sorted(by_type):
        s = by_type[t]
        acc = s["correct"] / s["total"] * 100 if s["total"] else 0
        rec = (sum(s["recall"]) / len(s["recall"]) * 100) if s["recall"] else 0
        lt = (sum(s["lat_turn"]) / len(s["lat_turn"])) if s["lat_turn"] else 0
        lq = (sum(s["lat_query"]) / len(s["lat_query"])) if s["lat_query"] else 0
        print(f"  {t:<32s}  {acc:5.1f}%  {rec:5.1f}%  {lt:7.2f}s  {lq:5.2f}s  ({s['total']} preg.)")


def show_failures(results, limit=10):

    fails = [r for r in results if not r.get("correct")][:limit]
    if not fails:
        print("Todas correctas.")
        return
    print(f"\n--- PRIMEROS {len(fails)} FALLOS ---")
    for r in fails:
        recall = r.get("recall_at_k")
        if recall is None:
            cause = "(sin label de sesión)"
        elif recall == 0:
            cause = "  ← FALLO DE RETRIEVE (no trajo turnos relevantes)"
        else:
            cause = "  ← FALLO DE ANSWER (retrieve OK pero LLM falló)"
        print(f"\n[{r['question_id']}] type={r.get('question_type')}  recall@k={recall}{cause}")
        print(f"  Q   : {r.get('question')}")
        print(f"  GOLD: {r.get('gold')}")
        print(f"  PRED: {r.get('prediction')}")
        print(f"  juez: {r.get('judge_verdict')} — {r.get('judge_reason')}")


c:\Users\HIlla\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:

N_QUESTIONS = 100
OUTPUT_PATH_100 = "eval_results_100_final.json"

subset_100 = select_balanced_subset(lme_data, n_total=N_QUESTIONS, seed=42)
print(f"Subset: {len(subset_100)} preguntas (seed=42)")
print(f"Distribución por tipo:")
for t, c in Counter(q["question_type"] for q in subset_100).most_common():
    print(f"  {t:35s} {c}")
print(f"\nConfiguración: k=10, embedding-large, score sim+r, read-only")

results_100 = run_eval(
    subset_100,
    output_path=OUTPUT_PATH_100,
    k=10,
    n_max=400,
    enable_conflict_resolution=True,
    resume=True,
)

print_stats(results_100)
show_failures(results_100, limit=10)


Subset: 100 preguntas (seed=42)
Distribución por tipo:
  temporal-reasoning                  18
  single-session-assistant            17
  multi-session                       17
  single-session-user                 16
  knowledge-update                    16
  single-session-preference           16

Configuración: k=10, embedding-large, score sim+r, read-only
[eval] total=100  ya hechas=0  pendientes=100


Eval: 100%|██████████| 100/100 [4:10:54<00:00, 150.54s/q, acc=57.0%, n=100, rec=100.0%]


MÉTRICAS  (100 preguntas)
  1) Accuracy (LLM-as-judge) :  57.0%  (57/100)
  2) Recall@k del retrieve   : 100.0%  (100 preg. con label)
  3) Latencia                : ingest/turno =  6.34s  |  query =  8.45s

Desglose por question_type:
  tipo                                 acc   rec@k  lat/turn   lat/q
  knowledge-update                   75.0%  100.0%     6.14s   2.91s  (16 preg.)
  multi-session                      41.2%  100.0%     9.06s   3.11s  (17 preg.)
  single-session-assistant           52.9%  100.0%     3.89s   3.58s  (17 preg.)
  single-session-preference          50.0%  100.0%     6.05s   3.26s  (16 preg.)
  single-session-user                81.2%  100.0%     6.03s   2.92s  (16 preg.)
  temporal-reasoning                 44.4%  100.0%     6.81s  32.52s  (18 preg.)

--- PRIMEROS 10 FALLOS ---

[0ddfec37] type=knowledge-update  recall@k=1  ← FALLO DE ANSWER (retrieve OK pero LLM falló)
  Q   : How many autographed baseballs have I added to my collection in the first thre